[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SivanSchwartz/representation-learning-course/blob/master/tutoring/session01/01_oop.ipynb)

# 01 · Classes: the Python underneath PyTorch and Lightning

This is code from lecture 1 (`lec1/pytorch_lightning/1. start code/simple_fc.py`). You've run it. By the end of this notebook, you'll be able to explain **every line**:

```python
class NN(nn.Module):                      # inheritance             → section 4
    def __init__(self, input_size, num_classes):
        super().__init__()                # calling the parent      → section 4
        self.fc1 = nn.Linear(input_size, 50)   # composition        → section 5
        self.fc2 = nn.Linear(50, num_classes)

    def forward(self, x):                 # a hook you override     → section 4
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = NN(784, 10)                       # an instance             → section 2
scores = model(x)                         # __call__ runs forward   → section 4
```

## 1. A class bundles data and behaviour

During training we keep averaging the loss over an epoch. Lightning does it for you when you write `self.log(..., on_epoch=True)`. Here's a class that does the same job by hand:

- **data** (attributes): `self.total` and `self.count`
- **behaviour** (methods): `update()` and `compute()`

In [ ]:
# ✍️ A class is a blueprint. __init__ runs once, when you create an object from it.

In [ ]:
# ✍️ Create an object (an "instance"), then call its methods

## 2. Two instances, two sets of data

In [ ]:
# 🤔 Predict: what do the two lines print?
val_loss = RunningMean()
val_loss.update(10.0)
print("train:", train_loss.compute())
print("val:  ", val_loss.compute())

Each instance has its **own** `total` and `count`. The class is the blueprint, and every instance is a separate object built from it.

This is why `model_a = NN(784, 10)` and `model_b = NN(784, 10)` are two models with two different sets of random weights. Training one doesn't touch the other.

## 3. What is `self`?

In [ ]:
# ✍️ These two lines do exactly the same thing. Python fills in `self` with the object before the dot.

In [ ]:
# 🤔 Predict: this method forgot `self`. What happens when we call it?
class Oops:
    def hello():
        return "hi"


Oops().hello()

Read the error **from the bottom line**: `takes 0 positional arguments but 1 was given`. We gave it nothing, so what was the 1? It was the object itself. Python always passes the instance as the first argument, so every method needs a `self` parameter to receive it.

### ✏️ Exercise 1: an `Accuracy` class (≈5 min)

Write an `Accuracy` class with the same shape as `RunningMean`:
- `__init__` starts two counters, `self.correct` and `self.total`, at 0.
- `update(preds, targets)` gets two equal-length lists. It adds the number of positions where `preds[i] == targets[i]` to `correct`, and the length of the lists to `total`.
- `compute()` returns `correct / total`.

`torchmetrics.Accuracy`, which lecture 1 uses in step 4, works the same way.

In [ ]:
# ✏️ Your code here
class Accuracy:
    def __init__(self):
        ...

    def update(self, preds, targets):
        ...

    def compute(self):
        ...

In [ ]:
# --- Check (do not edit) ---
acc = Accuracy()
acc.update([1, 2, 3], [1, 2, 0])
acc.update([7], [7])
result = acc.compute()
assert result is not None, "Finish the exercise above first ☝️ (compute() returned None)"
assert getattr(acc, "total", None) == 4, "After two updates with 3 + 1 predictions, self.total should be 4"
assert abs(result - 0.75) < 1e-9, f"Expected 0.75 (3 correct out of 4), got {result}"
print("✅ Correct! 3 out of 4 predictions match.")

## 4. Inheritance: reuse a parent class, override its hooks

Here's a tiny version of PyTorch's `nn.Module`. It does two jobs for every model built on it:
- it counts how many times the model was called;
- calling the object, as in `model(x)`, runs `forward(x)`. That's what `__call__` does: it makes an object callable like a function.

The parent doesn't know what your model computes, so its `forward` just raises an error. Each child class **overrides** it.

In [ ]:
# ✍️ The parent class

In [ ]:
# ✍️ A child class: `Double(TinyModule)` means "a Double is a TinyModule". It only writes forward.

In [ ]:
# 🤔 Predict: what does this print?
double(1)
double(2)
print(double.n_calls)

`Double` never defined `__init__`, `__call__` or `n_calls`. It **inherited** them from `TinyModule`. It only *overrode* `forward`, the one method the parent left for it to fill in.

That's the pattern behind `nn.Module` and Lightning: **the parent class calls a method that you write.** Methods like that are called *hooks*.

In [ ]:
# 🤔 Predict: what happens if we use the parent class directly?
TinyModule()(5)

### `super().__init__()`: let the parent set itself up

A child that needs its own `__init__` (here, to store `factor`) **replaces** the parent's. To run the parent's setup as well, it calls `super().__init__()`.

In [ ]:
# ✍️ super() means "the parent class"

In [ ]:
# 🤔 Predict: this child forgot super().__init__(). Which line fails, and why?
class BrokenScale(TinyModule):
    def __init__(self, factor):
        self.factor = factor

    def forward(self, x):
        return self.factor * x


broken = BrokenScale(3)
broken(5)

`'BrokenScale' object has no attribute 'n_calls'`: the parent's `__init__` never ran, so `self.n_calls` was never created. The object was half built. `nn.Module` keeps a lot more inside its `__init__`, so forgetting `super().__init__()` there breaks things straight away. You'll see its error message in notebook 02.

## 5. Composition: a model *has* layers

A class can store other objects as attributes. A model is built from layers this way, and the layers can contain layers of their own, so models form a tree.

In [ ]:
# ✍️ A model made of two smaller modules

In [ ]:
# 🤔 Predict: what does pipe(1) return? And after it runs, what are the three n_calls?
pipe = Pipeline()
print(pipe(1))
print(pipe.n_calls, pipe.first.n_calls, pipe.second.n_calls)

This is lec1's `self.fc1 = nn.Linear(...)`. `nn.Module` goes one step further: it notices every module you store on `self`, so `print(model)` shows the tree and `model.parameters()` finds every weight inside it. Notebook 02 shows both.

## 6. Dunder methods: making your object behave like a built-in

Methods with **d**ouble **under**scores, like `__init__` and `__call__`, are hooks that Python itself calls. Two more matter for PyTorch:
- `len(obj)` calls `obj.__len__()`
- `obj[i]` calls `obj.__getitem__(i)`

A PyTorch `Dataset` is exactly that: a class with these two methods. Let's load today's data: 6,000 MNIST digits for training and 1,000 for testing.

In [ ]:
import io
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/SivanSchwartz/representation-learning-course/master/tutoring/data/mnist_small.npz"


def load_mnist():
    """The local copy if there is one (your laptop), otherwise download it (Colab)."""
    for path in ["../data/mnist_small.npz", "../../data/mnist_small.npz"]:
        if os.path.exists(path):
            return dict(np.load(path))
    return dict(np.load(io.BytesIO(urllib.request.urlopen(DATA_URL).read())))


data = load_mnist()

In [ ]:
# ✍️ A dataset class: Python calls __len__ for len() and __getitem__ for [ ]

In [ ]:
# 🤔 Predict: what is train_ds[0]? What type is it, and what's inside?
image, label = train_ds[0]
plt.figure(figsize=(2, 2))
plt.imshow(image, cmap="gray")
plt.title(f"label = {label}")
plt.axis("off")
plt.show()

A PyTorch `DataLoader` only ever does two things with your dataset: it asks `len(dataset)`, and it asks `dataset[i]` for many `i`s, then stacks the answers into a batch. That's why lecture code can pass a torchvision dataset, a `TensorDataset`, or your own class: they all have `__len__` and `__getitem__`.

## Summary: lec1's `NN`, line by line

| Line | What it is |
|---|---|
| `class NN(nn.Module)` | **Inheritance**: `NN` gets everything `nn.Module` already does |
| `super().__init__()` | Run the parent's setup first, or the object is half built |
| `self.fc1 = nn.Linear(...)` | **Composition**: the model *has* layers |
| `def forward(self, x)` | A **hook**: the parent's `__call__` calls it |
| `model(x)` | `__call__` → `forward(x)` |
| `self` | The instance itself. Python passes it in for you. |

🔄 **Checkpoint:** the solution to this notebook is being released now. Run `git pull --no-edit`, then open `tutoring/session01/solutions/01_oop.ipynb` if you want to compare.